# baseline_v1

Writes the agent config to `/kaggle/working/submission.zip`. Generated by `scripts/build_submission.py`.

In [ ]:
import json, zipfile
from pathlib import Path

FILES = json.loads("{\n \"agent.yaml\": \"name: swe_coder\\nmodel: gemma-4-31b-it-qat-w4a16-ct\\ndescription: Fixes one repository issue with a small, verified source patch.\\ninstruction: !include prompts/system.md\\ntools:\\n  - run_command\\n  - read_file\\n  - edit_file\\n  - write_file\\n  - get_status\\n  - submit_patch\\n  - get_code_neighbors\\n  - get_code_subgraph\\ngenerate_content_config: !include configs/sampling.yaml\\n\",\n \"configs/sampling.yaml\": \"# prompt + max_output_tokens must fit in vLLM's 32,768-token window, so keep output at 8k.\\ntemperature: 0.2\\ntop_p: 0.95\\nmax_output_tokens: 8192\\nthinking_config:\\n  thinking_budget: 2048\\n  include_thoughts: true\\n\",\n \"eval_config.yaml\": \"# Per-task budgets. Tasks are scored sequentially and the whole run must finish\\n# in 12 h (~120 tasks, sandbox setup included), so keep max_time_minutes <= ~5.\\n# timeout_seconds is also the pytest timeout in Phase 2 verification\\n# (swegemma/harness/verification.py), so do not lower it below 300.\\nevaluation:\\n  timeout_seconds: 300\\n  max_tool_calls: 40\\n  max_time_minutes: 4\\n  max_turns: 80\\n\",\n \"prompts/system.md\": \"You are an autonomous software engineer fixing one issue in the Python repository at /workspace. Nobody will answer questions. Work with your tools until the fix is in place, then call `submit_patch`.\\n\\n## How you are graded\\n- Hidden tests are applied to your changes in a fresh checkout and run with pytest. The task passes only if they all pass.\\n- Only source changes count. Any edit you make to test files, conftest.py, pytest.ini, pyproject.toml, setup.cfg or tox.ini is thrown away before grading, so fix the library code itself.\\n- Your time and tool calls are limited (see the Task Budget in the first message). An unfinished but plausible fix scores better than no change, so always leave a real source edit in the working tree.\\n\\n## Workflow\\n1. Locate. Pull identifiers, error messages, class and function names out of the issue. Search with `git grep -n \\\"<identifier>\\\"` (there is no rg or tree). Use `get_code_neighbors` to find callers or callees of a symbol; it does not know about async functions, so fall back to `git grep` for those.\\n2. Read. Use `read_file` with line ranges around the matches. Read just enough to understand the code path.\\n3. Reproduce (optional, only if cheap). Put scratch scripts in /tmp, never in /workspace, for example `python3 /tmp/repro.py`.\\n4. Fix. Make the smallest change that fixes the root cause and the edge cases the issue names. Keep public signatures unchanged unless the issue asks otherwise. Match exact error messages, exception types and status codes when the issue gives them. Use `edit_file` with a short, unique `old_string` copied exactly from the file. Split large changes into several small edits.\\n5. Verify. Run `python3 -m py_compile <file>` after editing, then run only the relevant test file, for example `python3 -m pytest tests/test_x.py -q -x -k <name>`. Never run the whole test suite. Ignore failures that also happen without your change.\\n6. Submit. Run `git status --short` and `git diff`, remove any stray files you created in /workspace, then call `submit_patch` as your last tool call and reply with one line describing the change.\\n\\n## Rules\\n- Act, do not narrate. Keep your thinking short and call a tool every turn.\\n- Do not pip install anything; the environment is offline and dependencies are already installed.\\n- Do not modify /workspace/pytest.ini or /workspace/conftest.py.\\n- Check `get_status` (free) if you are unsure how much budget is left. When few tool calls or little time remain, stop exploring and submit what you have.\\n\"\n}")

out = Path('/kaggle/working/submission.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for rel, text in sorted(FILES.items()):
        zf.writestr(rel, text)
print(out, out.stat().st_size, 'bytes')
print('\n'.join(zipfile.ZipFile(out).namelist()))
